# Fine-tuning TinyLlama with LoRA

Loads TinyLlama-1.1B-Chat, fine-tunes it with LoRA on the curated training data, then exports the merged model to GGUF and registers it with Ollama for serving.

## Step 1: Load Configuration

In [14]:
import sys
from pathlib import Path

RUNEBOOK_ROOT = Path.cwd()  # notebook 01's clone cell already chdir'd here
if str(RUNEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(RUNEBOOK_ROOT))

from src.trainer import OllamaTrainer

trainer = OllamaTrainer(RUNEBOOK_ROOT / "config/training_config.yaml")
print(trainer.get_config_summary())


Fine-tuning Configuration:
  Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
  Epochs: 8
  Batch Size: 8
  Learning Rate: 0.0002
  LoRA: True



## Step 2: Run LoRA Fine-tuning

This trains on the single T4 GPU with the bank transaction dataset (800 train examples, 8 epochs, gradient accumulation 2 → effective batch 16). ~400 optimization steps, ~4-5 minutes.

In [15]:
# Colab's /content is the ephemeral runtime disk (no Databricks 500MB workspace file limit here)
trainer.output_dir = Path("/content/finetuning_output")
trainer.output_dir.mkdir(parents=True, exist_ok=True)

result = trainer.train()

print(f"Status: {result['status']}")
print(f"Final loss: {float(result['final_loss']) if isinstance(result['final_loss'], str) else result['final_loss']}")
print(f"Merged model saved to: {result['merged_dir']}")
print(f"LoRA adapter saved to: {result['adapter_dir']}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


: 

: 

: 

## Step 3: Plot the Training Loss Curve

In [16]:
import matplotlib.pyplot as plt

steps = [entry["step"] for entry in result["log_history"] if "loss" in entry]
losses = [entry["loss"] for entry in result["log_history"] if "loss" in entry]

plt.figure(figsize=(8, 4))
plt.plot(steps, losses, marker="o")
plt.xlabel("Step")
plt.ylabel("Training loss")
plt.title("LoRA fine-tuning loss")
plt.tight_layout()
plt.show()

TypeError: 'CompletedProcess' object is not subscriptable

## Step 4: Export to GGUF

Ollama can only serve GGUF models, so the merged fp16 model is converted via llama.cpp's `convert_hf_to_gguf.py`.

In [ ]:
from src.ollama_export import convert_to_gguf, create_ollama_model

merged_dir = Path(result["merged_dir"])
gguf_path = merged_dir / "model.gguf"

convert_to_gguf(
    merged_model_dir=merged_dir,
    output_gguf=gguf_path,
    llama_cpp_dir=Path("/content/llama.cpp"),
    outtype="q8_0",
)
print(f"\nGGUF model written to {gguf_path}")

## Step 5: Register the Model with Ollama

In [ ]:
OLLAMA_MODEL_NAME = "tinyllama-finetuned:latest"

create_ollama_model(
    name=OLLAMA_MODEL_NAME,
    gguf_path=gguf_path,
    system_prompt="You are a helpful assistant fine-tuned on domain-specific instructions.",
)
print(f"\nRegistered '{OLLAMA_MODEL_NAME}' with Ollama")

## Step 6: Sanity-check the Served Model

In [0]:
import requests

test_prompt = """Categorize the following bank transaction based on its description.
RAPPI*RAPPI COL CHAPINERO - $52,000 COP - 2024-06-15 21:10"""

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": OLLAMA_MODEL_NAME,
        "prompt": test_prompt,
        "stream": False,
    },
    timeout=5*60,
)
response.raise_for_status()
print("Prompt:", test_prompt.strip())
print("\n--- Model Response ---")
print(response.json()["response"])

Prompt: Categorize the following bank transaction based on its description.
RAPPI*RAPPI COL CHAPINERO - $52,000 COP - 2024-06-15 21:10

--- Model Response ---
Category: Food & Delivery. This is a food delivery or restaurant purchase. Amount of $52,000 COP charged on 2024-06-15 21:10.


## Fine-tuning & Deployment Complete

The fine-tuned model is trained, exported, and being served via Ollama. Next: run `04_evaluation.ipynb` to measure quality against the baseline.